# QREV v4.0.0 — Scientific finalization and freeze preparation

This notebook does not decode audio and does not recompute the four QREV analysis features. It verifies the completed cohort candidate, applies the accepted G10 roles, corrects the shorter-horizon sensitivity evidence from the saved boundary ledger, regenerates the audited figures, and prepares the immutable measurement freeze.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT_OVERRIDE = 'C:\\Users\\musikicn\\Desktop\\Nevena_project\\Paper_1\\paper_1'
RUN_PACKAGE_TESTS = True
RUN_FINALIZATION = True
SCIENTIFIC_REVIEW_DECISION = "ACCEPT_QREV_V400"
SCIENTIFIC_REVIEWER = "Nevena Musikic"
SCIENTIFIC_REVIEW_RATIONALE = (
    "Post-cohort analytical-validation review completed. Tail excess is retained as the primary conditional residual measurement; "
    "bounded persistence is secondary and non-independent; decay rate is exploratory conditional; normalized-fast SRMR is retained as a pinned established comparator. "
    "No QREV scalar or standalone gate is approved."
)
PUBLISH_AND_FREEZE = False

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "03_QREV":
    candidate = Path.cwd() / "notebooks reviewed" / "03_QREV"
    if candidate.exists():
        NOTEBOOK_DIR = candidate
PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE) if PROJECT_ROOT_OVERRIDE else NOTEBOOK_DIR.parents[1]
SOURCE_ROOT = PROJECT_ROOT / "outputs reviewed" / "reverberation" / "qrev-v4.0.0-candidate"
FINAL_ROOT = PROJECT_ROOT / "outputs reviewed" / "reverberation" / "qrev-v4.0.0"

for source_path in (PROJECT_ROOT / "src reviewed", PROJECT_ROOT / "src"):
    if str(source_path) not in sys.path:
        sys.path.insert(0, str(source_path))

from paper1_qc_reviewed.qrev_v400_final import (
    ACCEPTANCE_TOKEN,
    ANALYSIS_FEATURES,
    analysis_values_equal,
    finalize_candidate,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source candidate: {SOURCE_ROOT}")
print(f"Final candidate: {FINAL_ROOT}")
print(f"Decision: {SCIENTIFIC_REVIEW_DECISION}")


Project root: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1
Source candidate: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1\outputs reviewed\reverberation\qrev-v4.0.0-candidate
Final candidate: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1\outputs reviewed\reverberation\qrev-v4.0.0
Decision: ACCEPT_QREV_V400


In [2]:
if RUN_PACKAGE_TESTS:
    test_paths = [
        PROJECT_ROOT / "tests reviewed" / "test_qrev_v400.py",
        PROJECT_ROOT / "tests reviewed" / "test_qrev_v400_cohort.py",
        PROJECT_ROOT / "tests reviewed" / "test_qrev_v400_final.py",
    ]
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *map(str, test_paths), "-q", "--disable-warnings"],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError("Reviewed QREV tests failed")


..............................................                           [100%]



In [3]:
source_manifest_path = SOURCE_ROOT / "manifests" / "qrev_v400_cohort_candidate_manifest.json"
source_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
required = {
    "cohort_extraction_completed": True,
    "cohort_evidence_complete": True,
    "recording_count": 519,
    "participant_count": 224,
    "required_panels_complete": True,
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
}
for key, expected in required.items():
    observed = source_manifest.get(key)
    if observed != expected:
        raise RuntimeError(f"Source manifest mismatch for {key}: {observed!r}")
print("Completed QREV cohort candidate verified.")


Completed QREV cohort candidate verified.


In [4]:
provenance_files = [
    NOTEBOOK_DIR / "QREV_v400_FINAL_SCIENTIFIC_AUDIT.md",
    NOTEBOOK_DIR / "QREV_v400_FINAL_FEATURE_DECISIONS.csv",
    NOTEBOOK_DIR / "QREV_Family_Evaluation_Workbook_v1_0.docx",
    NOTEBOOK_DIR / "QREV_V4_0_0_FREEZE_CONTRACT.md",
    NOTEBOOK_DIR / "QREV_Validation_Checklist_v1_0.csv",
    NOTEBOOK_DIR / "QREV_Ten_Domain_Dashboard_v1_0.csv",
    NOTEBOOK_DIR / "QREV_V400_FINALIZATION_IMPLEMENTATION_REPORT.md",
]

if RUN_FINALIZATION:
    final_manifest = finalize_candidate(
        source_root=SOURCE_ROOT,
        final_root=FINAL_ROOT,
        scientific_review_decision=SCIENTIFIC_REVIEW_DECISION,
        scientific_reviewer=SCIENTIFIC_REVIEWER,
        scientific_review_rationale=SCIENTIFIC_REVIEW_RATIONALE,
        provenance_files=provenance_files,
    )
    print(json.dumps(final_manifest, indent=2))


{
  "measurement_version": "qrev-v4.0.0",
  "source_measurement_version": "qrev-v4.0.0-candidate",
  "family": "QREV",
  "family_display_name": "Reverberation / residual-tail manifestations",
  "candidate_only": true,
  "freeze_status": "ready_for_atomic_freeze",
  "freeze_allowed": true,
  "scientific_review_decision": "ACCEPT_QREV_V400",
  "scientific_reviewer": "Nevena Musikic",
  "scientific_review_rationale": "Post-cohort analytical-validation review completed. Tail excess is retained as the primary conditional residual measurement; bounded persistence is secondary and non-independent; decay rate is exploratory conditional; normalized-fast SRMR is retained as a pinned established comparator. No QREV scalar or standalone gate is approved.",
  "analysis_features": [
    "qrev_tail_excess_100ms_db",
    "qrev_tail_persistence_median_sec",
    "qrev_downward_decay_rate_db_per_sec",
    "qrev_srmr_norm"
  ],
  "primary_analysis_features": [
    "qrev_tail_excess_100ms_db"
  ],
  "secon

In [5]:
if RUN_FINALIZATION:
    source_features = pd.read_csv(SOURCE_ROOT / "tables" / "qrev_v400_analysis_features.csv")
    final_features = pd.read_csv(FINAL_ROOT / "tables" / "qrev_v400_analysis_features.csv")
    if not analysis_values_equal(source_features, final_features):
        raise RuntimeError("Finalization changed QREV analysis feature values")
    corrected_horizon = pd.read_csv(FINAL_ROOT / "validation" / "qrev_v400_corrected_horizon_sensitivity.csv")
    figure_index = pd.read_csv(FINAL_ROOT / "figures" / "qrev_v400_standardized_figure_index.csv")
    print(f"Numerical equivalence: True ({len(final_features)} recordings)")
    print(f"Corrected shorter-horizon rows: {len(corrected_horizon)}")
    print(f"Applicable figure bundles: {(figure_index.panel != 'I').sum()}")
    print(f"Panel I: {figure_index.loc[figure_index.panel.eq('I'), 'purpose'].item()}")


Numerical equivalence: True (519 recordings)
Corrected shorter-horizon rows: 192
Applicable figure bundles: 22
Panel I: no retained event detector


In [6]:
if RUN_FINALIZATION:
    final_manifest_path = FINAL_ROOT / "manifests" / "qrev_v400_final_candidate_manifest.json"
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
    if SCIENTIFIC_REVIEW_DECISION == ACCEPTANCE_TOKEN and not final_manifest.get("freeze_allowed"):
        raise RuntimeError("Accepted candidate did not authorize atomic freeze")
    if PUBLISH_AND_FREEZE:
        raise RuntimeError("This notebook prepares but does not execute the atomic freeze")
    print("QREV v4.0.0 FINALIZATION COMPLETE")
    print("Candidate is ready for atomic freeze." if final_manifest.get("freeze_allowed") else "Scientific acceptance remains pending.")


QREV v4.0.0 FINALIZATION COMPLETE
Candidate is ready for atomic freeze.
